# 03 · Senescence scoringSenePy scoring · reference-anchored threshold · depth diagnosticEach block is **Why → Test → Display**. Code is lifted from `module1p1.ipynb` with the source cell number recorded; anything not traceable to source is marked `⟨NEW⟩`.

## Configuration**Why.** One config module resolves every path and parameter from a single dataset key, so the only thing that differs between cohorts is that key. Merged from the dataset dicts already present in the source notebooks.

In [ ]:
from pathlib import Pathimport warnings; warnings.filterwarnings('ignore')from config import CFG, assert_layersimport config as CDATASET = 'psychad_aging'      # <<< the only line you changecfg = CFG.for_dataset(DATASET)cfg.echo()def why(block, question, rationale=None):    """Print the Why so it lands in executed output, not only in markdown."""    print("\n" + "=" * 78)    print(f"  {block}")    print("=" * 78)    print(f"  Q: {question}")    if rationale:        for line in rationale.split(" | "):            print(f"     {line}")    print()def gate(label, ok, detail=""):    """Fail loudly. A failed gate stops the module rather than flowing downstream."""    mark = "OK  " if ok else "FAIL"    print(f"  [{mark}] {label}{'  — ' + detail if detail else ''}")    if not ok:        raise AssertionError(f"GATE FAILED: {label}. {detail}")    return ok

## Setup**Why.** SenePy import check.<sub>source: `module1p1.ipynb` cells 3, 4</sub>

In [ ]:
why("Setup", "SenePy import check")

In [ ]:
# ── source: module1p1.ipynb cell 3 ──import scanpy as scimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom pathlib import Pathimport warningswarnings.filterwarnings('ignore')# SenePy for senescence scoringtry:    import senepy    print("✓ senepy installed")except ImportError:    print("⚠ senepy not installed. Run: pip install senepy")    raiseprint(f"scanpy: {sc.__version__}")print(f"numpy: {np.__version__}")print(f"pandas: {pd.__version__}")

In [ ]:
# ── source: module1p1.ipynb cell 4 ──# Figure settingsplt.rcParams.update({    'figure.dpi': 150,    'savefig.dpi': 300,    'font.size': 10,    'axes.labelsize': 10,    'axes.titlesize': 11,    'legend.fontsize': 9,    'font.family': 'sans-serif',    'axes.linewidth': 1.0,    'axes.grid': False,    'pdf.fonttype': 42,})sc.settings.verbosity = 1sc.settings.set_figure_params(dpi=150, dpi_save=300, facecolor='white', frameon=False)print("✓ Figure settings configured")

## Load Data and Create Study Groups**Why.** Builds `Study_Group`, which harmonizes aging cohorts (decade bins) and disease cohorts (diagnosis) into one column, so the same threshold logic serves both modes.<sub>source: `module1p1.ipynb` cell 8</sub>

In [ ]:
why("Load Data and Create Study Groups", "Builds `Study_Group`, which harmonizes aging cohorts (decade bins) and disease cohorts (diagnosis) into one column, so the same threshold logic serves both modes")

In [ ]:
# ── source: module1p1.ipynb cell 8 ──print("\n" + "="*80)print("LOADING DATA")print("="*80)adata = sc.read_h5ad(INPUT_FILE)print(f"\nLoaded: {INPUT_FILE.name}")print(f"  Cells: {adata.n_obs:,}")print(f"  Genes: {adata.n_vars:,}")# Verify required columns existrequired_cols = [CELL_TYPE_COLUMN, AGE_COLUMN, SEX_COLUMN, DONOR_COLUMN]missing_cols = [col for col in required_cols if col not in adata.obs.columns]if missing_cols:    print(f"\n⚠ Warning: Missing columns: {missing_cols}")    print("Available columns:", list(adata.obs.columns))print("\n" + "="*80)print("CREATING STUDY_GROUP COLUMN")print("="*80)if config['type'] == 'aging':    # Create age bins    print(f"\nCreating age bins...")    age_bins = config['age_bins']        def assign_age_group(age):        for bin_min, bin_max in age_bins:            if bin_min <= age <= bin_max:                return f"Age_{bin_min}_{bin_max}"        return "Age_Unknown"        adata.obs[STUDY_GROUP_COLUMN] = adata.obs[AGE_COLUMN].apply(assign_age_group)        # Show distribution    group_counts = adata.obs[STUDY_GROUP_COLUMN].value_counts().sort_index()    print(f"\nAge group distribution:")    for group, count in group_counts.items():        print(f"  {group}: {count:,} cells")    elif config['type'] == 'disease':    # Use diagnosis column    diag_col = config['diagnosis_column']    if diag_col and diag_col in adata.obs.columns:        print(f"\nUsing diagnosis groups from '{diag_col}'...")        adata.obs[STUDY_GROUP_COLUMN] = adata.obs[diag_col].astype(str)                # Show distribution        group_counts = adata.obs[STUDY_GROUP_COLUMN].value_counts()        print(f"\nDiagnosis group distribution:")        for group, count in group_counts.items():            print(f"  {group}: {count:,} cells")    else:        print(f"\n⚠ Diagnosis column '{diag_col}' not found!")        print(f"Available columns: {list(adata.obs.columns)}")# Verify reference group existsif REFERENCE_GROUP not in adata.obs[STUDY_GROUP_COLUMN].unique():    print(f"\n⚠ WARNING: Reference group '{REFERENCE_GROUP}' not found in Study_Group!")    print(f"Available groups: {sorted(adata.obs[STUDY_GROUP_COLUMN].unique())}")else:    n_ref = (adata.obs[STUDY_GROUP_COLUMN] == REFERENCE_GROUP).sum()    print(f"\n✓ Reference group '{REFERENCE_GROUP}': {n_ref:,} cells")print("\n✓ Data loaded and Study_Group created")

## SenePy Senescence Scoring**Why.** SenePy is the pre-specified primary panel because it is built for scRNA-seq with cell-type and sex-specific calibration, unlike the bulk-fibroblast-derived alternatives. `identifiers=[cell_type, sex]` is what makes the scoring cell-type and sex-specific. Hippocampus hubs are used as the closest available brain calibration — SenePy has no cortical hub, and this approximation must be stated in Methods.<sub>source: `module1p1.ipynb` cell 10</sub>

In [ ]:
why("SenePy Senescence Scoring", "SenePy is the pre-specified primary panel because it is built for scRNA-seq with cell-type and sex-specific calibration, unlike the bulk-fibroblast-derived alternatives")

In [ ]:
# ── source: module1p1.ipynb cell 10 ──print("\n" + "="*80)print("SENESCENCE SCORING")print("="*80)print(f"\nScoring parameters:")print(f"  Species: Human")print(f"  Tissue: {SENEPY_TISSUE}")print(f"  Cell type column: {CELL_TYPE_COLUMN}")print(f"  Sex column: {SEX_COLUMN}")# Step 1: Load SenePy hubsprint("\n1. Loading SenePy hubs...")import senepy as sphubs = sp.load_hubs(species='Human')print(f"   ✓ Loaded {len(hubs.metadata)} hub entries")# Step 2: Filter for brain tissueprint(f"\n2. Filtering for {SENEPY_TISSUE} tissue...")brain_hubs = hubs.metadata[hubs.metadata.tissue == SENEPY_TISSUE]print(f"   ✓ Found {len(brain_hubs)} {SENEPY_TISSUE}-specific hubs")# Step 3: Merge hubsprint(f"\n3. Merging hubs...")hubs.merge_hubs(brain_hubs, new_name='brain')print(f"   ✓ Hubs merged")# Step 4: Create translatorprint(f"\n4. Creating gene translator...")translator = sp.translator(hub=hubs.hubs, data=adata)print(f"   ✓ Translator created")# Step 5: Score all cellsprint(f"\n5. Scoring cells (cell-type and sex-specific)...")print(f"   This may take 10-20 minutes for large datasets...")adata.obs['senescence_score'] = sp.score_all_cells(    adata,     hubs.hubs['brain'],    identifiers=[CELL_TYPE_COLUMN, SEX_COLUMN],    translator=translator)print(f"\n✓ Senescence scoring complete")# Show score statisticsscores = adata.obs['senescence_score']print(f"\nScore statistics:")print(f"  Mean: {scores.mean():.3f}")print(f"  Median: {scores.median():.3f}")print(f"  Std: {scores.std():.3f}")print(f"  Min: {scores.min():.3f}")print(f"  Max: {scores.max():.3f}")

## Define Senescence Threshold**Why.** Mean + 2SD of the REFERENCE GROUP only, per cell type — youngest bin for aging, control for disease. A threshold computed on pooled data would slide with the very effect being measured, partially erasing it. Note the fallback path when a cell type has no reference cells pools across all groups; that is a silent deviation and is now gated.<sub>source: `module1p1.ipynb` cell 13</sub>

In [ ]:
why("Define Senescence Threshold", "Mean + 2SD of the REFERENCE GROUP only, per cell type — youngest bin for aging, control for disease")

In [ ]:
# ── source: module1p1.ipynb cell 13 ──print("\n" + "="*80)print("SENESCENCE THRESHOLD DEFINITION")print("="*80)print(f"\nReference group: {REFERENCE_GROUP}")print(f"Threshold: Mean + {SD_THRESHOLD} SD per cell type")# Get reference group cellsref_mask = adata.obs[STUDY_GROUP_COLUMN] == REFERENCE_GROUPn_ref = ref_mask.sum()if n_ref == 0:    raise ValueError(f"Reference group '{REFERENCE_GROUP}' not found!")print(f"\nReference cells: {n_ref:,}")# Calculate threshold per cell typethresholds = {}print(f"\nCalculating thresholds per cell type...")for cell_type in adata.obs[CELL_TYPE_COLUMN].unique():    # Get reference cells of this type    ref_ct_mask = ref_mask & (adata.obs[CELL_TYPE_COLUMN] == cell_type)    ref_scores = adata.obs.loc[ref_ct_mask, 'senescence_score']        if len(ref_scores) > 0:        mean = ref_scores.mean()        std = ref_scores.std()        threshold = mean + (SD_THRESHOLD * std)        thresholds[cell_type] = threshold                print(f"  {cell_type}:")        print(f"    n={len(ref_scores):,}, mean={mean:.3f}, std={std:.3f}, threshold={threshold:.3f}")    else:        print(f"  {cell_type}: No reference cells - using global threshold")        thresholds[cell_type] = adata.obs['senescence_score'].mean() + (SD_THRESHOLD * adata.obs['senescence_score'].std())# Apply thresholds to classify cellsprint(f"\nClassifying cells as SnC (senescent) or Non-SnC...")adata.obs['is_senescent'] = Falsefor cell_type, threshold in thresholds.items():    mask = (adata.obs[CELL_TYPE_COLUMN] == cell_type) & (adata.obs['senescence_score'] >= threshold)    adata.obs.loc[mask, 'is_senescent'] = True# Add binary labeladata.obs['senescence_label'] = adata.obs['is_senescent'].map({True: 'SnC', False: 'Non-SnC'})# Show resultsn_snc = adata.obs['is_senescent'].sum()pct_snc = n_snc / len(adata.obs) * 100print(f"\n✓ Classification complete")print(f"\nOverall results:")print(f"  SnC (senescent): {n_snc:,} cells ({pct_snc:.1f}%)")print(f"  Non-SnC: {len(adata.obs) - n_snc:,} cells ({100-pct_snc:.1f}%)")print(f"\nSnC proportion by cell type:")for cell_type in sorted(adata.obs[CELL_TYPE_COLUMN].unique()):    ct_mask = adata.obs[CELL_TYPE_COLUMN] == cell_type    ct_snc = (ct_mask & adata.obs['is_senescent']).sum()    ct_total = ct_mask.sum()    ct_pct = ct_snc / ct_total * 100 if ct_total > 0 else 0    print(f"  {cell_type}: {ct_snc:,}/{ct_total:,} ({ct_pct:.1f}%)")

## UMI Confounding Diagnostic**Why.** The decisive check for module 02. Residuals decouple expression from depth, but the question is whether the SCORE is decoupled. Reported, not corrected: senescent cells are larger and transcriptionally hyperactive, so elevated UMI is a predicted biological property and regressing it out again would remove real signal. `senescence_score_adjusted` is computed as a diagnostic and is deliberately NOT consumed downstream.<sub>source: `module1p1.ipynb` cells 15, 16, 17, 18, 19</sub>

In [ ]:
why("UMI Confounding Diagnostic", "The decisive check for module 02")

In [ ]:
# ── source: module1p1.ipynb cell 15 ──# ═══════════════════════════════════════════════════════════════════════════════# UMI DISTRIBUTION: SnC vs Non-SnC# ═══════════════════════════════════════════════════════════════════════════════print("\n" + "="*80)print("UMI DISTRIBUTION: SnC vs Non-SnC")print("="*80)from scipy.stats import mannwhitneyu# ColorsCOLOR_SNC = '#E15759'COLOR_NONSNC = '#4E79A7'CELL_TYPE_COLORS = {    'Excitatory': '#4E79A7', 'Inhibitory': '#F28E2B', 'Astrocyte': '#E15759',    'Oligodendrocyte': '#76B7B2', 'OPC': '#59A14F', 'Microglia': '#EDC948',    'Endothelial': '#B07AA1', 'Pericyte': '#FF9DA7', 'VSMC': '#9C755F',    'VLMC': '#BAB0AC', 'PVM': '#D37295', 'Adaptive': '#FABFD2',}snc_mask = adata.obs['is_senescent'] == Truenonsnc_mask = adata.obs['is_senescent'] == Falsesnc_umi = adata.obs.loc[snc_mask, 'total_counts']nonsnc_umi = adata.obs.loc[nonsnc_mask, 'total_counts']# ─────────────────────────────────────────────────────────────────────────────# A. Overall Statistics# ─────────────────────────────────────────────────────────────────────────────print(f"\n{'─'*60}")print("A. OVERALL STATISTICS")print(f"{'─'*60}")fold_median = snc_umi.median() / nonsnc_umi.median()fold_mean = snc_umi.mean() / nonsnc_umi.mean()stat, p_val = mannwhitneyu(snc_umi, nonsnc_umi, alternative='two-sided')print(f"\n  {'Metric':<15} {'Non-SnC':>12} {'SnC':>12} {'Fold':>10}")print(f"  {'-'*52}")print(f"  {'N Cells':<15} {len(nonsnc_umi):>12,} {len(snc_umi):>12,}")print(f"  {'Min UMI':<15} {nonsnc_umi.min():>12,.0f} {snc_umi.min():>12,.0f}")print(f"  {'Q25 UMI':<15} {nonsnc_umi.quantile(0.25):>12,.0f} {snc_umi.quantile(0.25):>12,.0f}")print(f"  {'Median UMI':<15} {nonsnc_umi.median():>12,.0f} {snc_umi.median():>12,.0f} {fold_median:>9.2f}x")print(f"  {'Q75 UMI':<15} {nonsnc_umi.quantile(0.75):>12,.0f} {snc_umi.quantile(0.75):>12,.0f}")print(f"  {'Max UMI':<15} {nonsnc_umi.max():>12,.0f} {snc_umi.max():>12,.0f}")print(f"  {'Mean UMI':<15} {nonsnc_umi.mean():>12,.0f} {snc_umi.mean():>12,.0f} {fold_mean:>9.2f}x")print(f"\n  Mann-Whitney U test: p = {p_val:.2e}")if fold_median > 1.5:    print(f"\n  ⚠ SnC cells have {fold_median:.1f}x higher median UMI than Non-SnC")    print(f"    → Strong technical confounding detected")else:    print(f"\n  ✓ UMI fold difference within acceptable range (≤1.5x)")# ─────────────────────────────────────────────────────────────────────────────# B. Per Cell Type Statistics# ─────────────────────────────────────────────────────────────────────────────print(f"\n{'─'*60}")print("B. PER CELL TYPE STATISTICS")print(f"{'─'*60}")ct_order = adata.obs[CELL_TYPE_COLUMN].value_counts().index.tolist()print(f"\n  {'Cell Type':<16} {'Non-SnC Med':>12} {'SnC Med':>12} {'Fold':>8} {'Flag':>6}")print(f"  {'-'*58}")ct_stats = []for ct in ct_order:    ct_snc = adata.obs.loc[(adata.obs[CELL_TYPE_COLUMN] == ct) & snc_mask, 'total_counts']    ct_nonsnc = adata.obs.loc[(adata.obs[CELL_TYPE_COLUMN] == ct) & nonsnc_mask, 'total_counts']        if len(ct_snc) > 0 and len(ct_nonsnc) > 0:        ct_fold = ct_snc.median() / ct_nonsnc.median()    else:        ct_fold = np.nan        flag = '⚠' if ct_fold > 1.5 else ''        ct_stats.append({        'Cell_Type': ct,        'N_NonSnC': len(ct_nonsnc),        'N_SnC': len(ct_snc),        'Median_NonSnC': ct_nonsnc.median() if len(ct_nonsnc) > 0 else np.nan,        'Median_SnC': ct_snc.median() if len(ct_snc) > 0 else np.nan,        'Fold': ct_fold    })        med_nonsnc = ct_nonsnc.median() if len(ct_nonsnc) > 0 else np.nan    med_snc = ct_snc.median() if len(ct_snc) > 0 else np.nan        print(f"  {ct:<16} {med_nonsnc:>12,.0f} {med_snc:>12,.0f} {ct_fold:>7.2f}x {flag:>6}")ct_stats_df = pd.DataFrame(ct_stats)n_flagged = (ct_stats_df['Fold'] > 1.5).sum()if n_flagged > 0:    print(f"\n  ⚠ {n_flagged} cell types with fold > 1.5x")else:    print(f"\n  ✓ No cell types with extreme UMI differences")# ─────────────────────────────────────────────────────────────────────────────# C. Overall Plots# ─────────────────────────────────────────────────────────────────────────────print(f"\n{'─'*60}")print("C. OVERALL PLOTS")print(f"{'─'*60}")fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))for ax in axes:    ax.spines['top'].set_visible(False)    ax.spines['right'].set_visible(False)# Panel 1: Histogrambins = np.linspace(np.log10(adata.obs['total_counts'].min() + 1),                    np.log10(adata.obs['total_counts'].max() + 1), 50)axes[0].hist(np.log10(nonsnc_umi + 1), bins=bins, color=COLOR_NONSNC,              alpha=0.6, density=True, label=f'Non-SnC (n={len(nonsnc_umi):,})')axes[0].hist(np.log10(snc_umi + 1), bins=bins, color=COLOR_SNC,              alpha=0.6, density=True, label=f'SnC (n={len(snc_umi):,})')axes[0].axvline(np.log10(nonsnc_umi.median() + 1), color=COLOR_NONSNC,                 linestyle='--', linewidth=2, label=f'Non-SnC med: {nonsnc_umi.median():,.0f}')axes[0].axvline(np.log10(snc_umi.median() + 1), color=COLOR_SNC,                 linestyle='--', linewidth=2, label=f'SnC med: {snc_umi.median():,.0f}')axes[0].set_xlabel('log₁₀(UMI + 1)')axes[0].set_ylabel('Density')axes[0].set_title(f'UMI Distribution (fold = {fold_median:.2f}x)')axes[0].legend(fontsize=7, frameon=False, loc='upper right')# Panel 2: Box plotbox_data = [nonsnc_umi.values, snc_umi.values]bp = axes[1].boxplot(box_data, positions=[0, 1], widths=0.6, patch_artist=True, showfliers=False)bp['boxes'][0].set_facecolor(COLOR_NONSNC)bp['boxes'][1].set_facecolor(COLOR_SNC)for box in bp['boxes']:    box.set_alpha(0.7)axes[1].set_xticks([0, 1])axes[1].set_xticklabels(['Non-SnC', 'SnC'])axes[1].set_ylabel('UMI Counts')axes[1].set_title('UMI by SnC Status')# Add median annotationsfor i, (data, color) in enumerate(zip(box_data, [COLOR_NONSNC, COLOR_SNC])):    med = np.median(data)    axes[1].text(i, med, f'{med:,.0f}', ha='center', va='bottom', fontsize=8, color='black')# Panel 3: Score vs UMI colored by SnC statusaxes[2].scatter(nonsnc_umi, adata.obs.loc[nonsnc_mask, 'senescence_score'],                 s=1, alpha=0.05, c=COLOR_NONSNC, label='Non-SnC', rasterized=True)axes[2].scatter(snc_umi, adata.obs.loc[snc_mask, 'senescence_score'],                 s=1, alpha=0.1, c=COLOR_SNC, label='SnC', rasterized=True)axes[2].set_xlabel('Total UMI')axes[2].set_ylabel('Senescence Score')axes[2].set_title('Score vs UMI by SnC Status')axes[2].legend(fontsize=8, frameon=False, markerscale=5)plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_umi_snc_comparison_overall.svg', dpi=300, bbox_inches='tight')plt.savefig(FIGURES_DIR / f'{DATASET}_umi_snc_comparison_overall.svg', dpi=150, bbox_inches='tight')plt.show()print(f"\n✓ Saved: {DATASET}_umi_snc_comparison_overall.svg")# ─────────────────────────────────────────────────────────────────────────────# D. Per Cell Type Violin Plots# ─────────────────────────────────────────────────────────────────────────────print(f"\n{'─'*60}")print("D. PER CELL TYPE VIOLIN PLOTS")print(f"{'─'*60}")n_ct = len(ct_order)fig, ax = plt.subplots(figsize=(max(10, n_ct * 0.8), 4))ax.spines['top'].set_visible(False)ax.spines['right'].set_visible(False)positions_nonsnc = np.arange(n_ct) * 2positions_snc = np.arange(n_ct) * 2 + 0.7# Non-SnC violinsnonsnc_data = [np.log10(adata.obs[(adata.obs[CELL_TYPE_COLUMN] == ct) & nonsnc_mask]['total_counts'].values + 1)                for ct in ct_order]vp1 = ax.violinplot(nonsnc_data, positions=positions_nonsnc, showmedians=True, showextrema=False, widths=0.6)for body in vp1['bodies']:    body.set_facecolor(COLOR_NONSNC)    body.set_alpha(0.7)vp1['cmedians'].set_color('black')# SnC violinssnc_data = [np.log10(adata.obs[(adata.obs[CELL_TYPE_COLUMN] == ct) & snc_mask]['total_counts'].values + 1)             for ct in ct_order]vp2 = ax.violinplot(snc_data, positions=positions_snc, showmedians=True, showextrema=False, widths=0.6)for body in vp2['bodies']:    body.set_facecolor(COLOR_SNC)    body.set_alpha(0.7)vp2['cmedians'].set_color('black')# X-axisax.set_xticks(np.arange(n_ct) * 2 + 0.35)ax.set_xticklabels(ct_order, rotation=45, ha='right', fontsize=9)ax.set_ylabel('log₁₀(UMI + 1)')ax.set_title('UMI Distribution by Cell Type and SnC Status')# Legendfrom matplotlib.patches import Patchlegend_elements = [Patch(facecolor=COLOR_NONSNC, alpha=0.7, label='Non-SnC'),                   Patch(facecolor=COLOR_SNC, alpha=0.7, label='SnC')]ax.legend(handles=legend_elements, loc='upper right', frameon=False)# Add fold annotationsfor i, ct in enumerate(ct_order):    fold = ct_stats_df[ct_stats_df['Cell_Type'] == ct]['Fold'].values[0]    if not np.isnan(fold):        y_pos = max(np.max(nonsnc_data[i]), np.max(snc_data[i])) + 0.1        color = '#E15759' if fold > 1.5 else '#333333'        ax.text(positions_nonsnc[i] + 0.35, y_pos, f'{fold:.1f}x',                 ha='center', va='bottom', fontsize=7, color=color)plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_umi_snc_comparison_by_celltype.svg', dpi=300, bbox_inches='tight')plt.savefig(FIGURES_DIR / f'{DATASET}_umi_snc_comparison_by_celltype.svg', dpi=150, bbox_inches='tight')plt.show()print(f"✓ Saved: {DATASET}_umi_snc_comparison_by_celltype.svg")# ─────────────────────────────────────────────────────────────────────────────# E. Summary# ─────────────────────────────────────────────────────────────────────────────print("\n" + "="*80)print("UMI CONFOUNDING SUMMARY (SnC vs Non-SnC)")print("="*80)print(f"\n  Overall UMI fold (SnC/Non-SnC): {fold_median:.2f}x")print(f"  Cell types with fold > 1.5x: {n_flagged}/{len(ct_order)}")if fold_median > 1.5 or n_flagged > 0:    print(f"\n  ⚠ CONFIRMATION: UMI strongly confounds SnC classification")    print(f"    → Module 02 models MUST include log10(total_counts) as covariate")else:    print(f"\n  ✓ UMI confounding within acceptable limits")print("="*80)

In [ ]:
# ── source: module1p1.ipynb cell 16 ──# ════════════════════════════════════════════════════════════════════════════════# UMI REGRESSION FROM SENESCENCE SCORES# ════════════════════════════════════════════════════════════════════════════════from sklearn.linear_model import LinearRegressionfrom scipy import stats  # <-- This was missingprint("\n" + "="*80)print("UMI REGRESSION FROM SENESCENCE SCORES")print("="*80)# ────────────────────────────────────────────────────────────────────────────────# CONFIGURATION# ────────────────────────────────────────────────────────────────────────────────SCORE_COL = 'senescence_score'ADJUSTED_SCORE_COL = 'senescence_score_adjusted'UMI_COL = 'total_counts'MIN_CELLS_REGRESSION = 50print(f"\nConfiguration:")print(f"  Score column: {SCORE_COL}")print(f"  Cell type column: {CELL_TYPE_COLUMN}")print(f"  Reference group: {REFERENCE_GROUP}")print(f"  Threshold: Mean + {SD_THRESHOLD} SD")# ────────────────────────────────────────────────────────────────────────────────# ENSURE UMI COUNTS EXIST# ────────────────────────────────────────────────────────────────────────────────if UMI_COL not in adata.obs.columns:    print(f"\n⚠ '{UMI_COL}' not in adata.obs")    if 'counts' in adata.layers:        print(f"  → Calculating from layers['counts']...")        adata.obs[UMI_COL] = np.array(adata.layers['counts'].sum(axis=1)).flatten()        print(f"  ✓ {UMI_COL} calculated")    else:        raise ValueError(f"Cannot find UMI counts")# ────────────────────────────────────────────────────────────────────────────────# STEP 1: PER CELL-TYPE REGRESSION# ────────────────────────────────────────────────────────────────────────────────print(f"\n" + "-"*60)print("STEP 1: Per Cell-Type Regression")print("-"*60)adata.obs[ADJUSTED_SCORE_COL] = np.nanregression_stats = []for ct in sorted(adata.obs[CELL_TYPE_COLUMN].unique()):    mask = adata.obs[CELL_TYPE_COLUMN] == ct    n_cells = mask.sum()        if n_cells < MIN_CELLS_REGRESSION:        print(f"  {ct}: SKIPPED (n={n_cells:,} < {MIN_CELLS_REGRESSION})")        adata.obs.loc[mask, ADJUSTED_SCORE_COL] = adata.obs.loc[mask, SCORE_COL]        regression_stats.append({            'Cell_Type': ct, 'N_Cells': n_cells, 'Status': 'skipped',            'R_squared': np.nan, 'Slope': np.nan, 'P_value': np.nan        })        continue        X = np.log10(adata.obs.loc[mask, UMI_COL].values).reshape(-1, 1)    y = adata.obs.loc[mask, SCORE_COL].values        valid = np.isfinite(X.flatten()) & np.isfinite(y)    X_valid, y_valid = X[valid], y[valid]        model = LinearRegression().fit(X_valid, y_valid)        residuals = y - model.predict(X)    adjusted = residuals + y.mean()    adata.obs.loc[mask, ADJUSTED_SCORE_COL] = adjusted        y_pred = model.predict(X_valid)    ss_res = np.sum((y_valid - y_pred) ** 2)    ss_tot = np.sum((y_valid - y_valid.mean()) ** 2)    r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0    r, p_value = stats.pearsonr(X_valid.flatten(), y_valid)        regression_stats.append({        'Cell_Type': ct, 'N_Cells': n_cells, 'Status': 'adjusted',        'R_squared': r_squared, 'Slope': model.coef_[0],         'Intercept': model.intercept_, 'P_value': p_value,        'Original_Mean': y.mean(), 'Adjusted_Mean': adjusted.mean()    })        print(f"  {ct}: n={n_cells:,}, R²={r_squared:.3f}, slope={model.coef_[0]:.4f}, p={p_value:.2e}")regression_df = pd.DataFrame(regression_stats)mean_r2 = regression_df[regression_df['Status'] == 'adjusted']['R_squared'].mean()print(f"\n✓ Regression complete | Mean R² = {mean_r2:.3f}")# ────────────────────────────────────────────────────────────────────────────────# STEP 2: CALCULATE ADJUSTED THRESHOLDS# ────────────────────────────────────────────────────────────────────────────────print(f"\n" + "-"*60)print("STEP 2: Calculate Adjusted Thresholds")print("-"*60)print(f"Reference group: {REFERENCE_GROUP}")ref_mask = adata.obs[STUDY_GROUP_COLUMN] == REFERENCE_GROUPthresholds_adjusted = {}for ct in sorted(adata.obs[CELL_TYPE_COLUMN].unique()):    ref_ct_mask = ref_mask & (adata.obs[CELL_TYPE_COLUMN] == ct)    ref_scores = adata.obs.loc[ref_ct_mask, ADJUSTED_SCORE_COL].dropna()        if len(ref_scores) >= 10:        mean = ref_scores.mean()        std = ref_scores.std()        threshold = mean + (SD_THRESHOLD * std)        thresholds_adjusted[ct] = threshold        print(f"  {ct}: n={len(ref_scores):,}, threshold={threshold:.3f}")    else:        ct_scores = adata.obs.loc[adata.obs[CELL_TYPE_COLUMN] == ct, ADJUSTED_SCORE_COL].dropna()        threshold = ct_scores.mean() + (SD_THRESHOLD * ct_scores.std())        thresholds_adjusted[ct] = threshold        print(f"  {ct}: using global (threshold={threshold:.3f})")# ────────────────────────────────────────────────────────────────────────────────# STEP 3: APPLY ADJUSTED CLASSIFICATION# ────────────────────────────────────────────────────────────────────────────────print(f"\n" + "-"*60)print("STEP 3: Apply Adjusted Classification")print("-"*60)adata.obs['is_senescent_adjusted'] = Falsefor ct, threshold in thresholds_adjusted.items():    mask = (adata.obs[CELL_TYPE_COLUMN] == ct) & (adata.obs[ADJUSTED_SCORE_COL] >= threshold)    adata.obs.loc[mask, 'is_senescent_adjusted'] = Trueadata.obs['senescence_label_adjusted'] = adata.obs['is_senescent_adjusted'].map(    {True: 'SnC', False: 'Non-SnC'})print("✓ Classification complete")# ════════════════════════════════════════════════════════════════════════════════# STEP 4: MULTI-LEVEL COMPARISON# ════════════════════════════════════════════════════════════════════════════════print(f"\n" + "="*80)print("COMPARISON: ORIGINAL vs ADJUSTED")print("="*80)# ────────────────────────────────────────────────────────────────────────────────# A. OVERALL# ────────────────────────────────────────────────────────────────────────────────print(f"\n" + "-"*60)print("A. OVERALL")print("-"*60)total = len(adata.obs)orig_snc = adata.obs['is_senescent'].sum()adj_snc = adata.obs['is_senescent_adjusted'].sum()orig_score_mean = adata.obs[SCORE_COL].mean()adj_score_mean = adata.obs[ADJUSTED_SCORE_COL].mean()print(f"\n  {'Metric':<25} {'Original':<15} {'Adjusted':<15} {'Δ':<15}")print("  " + "-"*70)print(f"  {'Mean Score':<25} {orig_score_mean:<15.4f} {adj_score_mean:<15.4f} {adj_score_mean - orig_score_mean:<+15.4f}")print(f"  {'N SnC':<25} {orig_snc:<15,} {adj_snc:<15,} {adj_snc - orig_snc:<+15,}")print(f"  {'% SnC':<25} {orig_snc/total*100:<15.2f} {adj_snc/total*100:<15.2f} {(adj_snc-orig_snc)/total*100:<+15.2f}")# ────────────────────────────────────────────────────────────────────────────────# B. PER STUDY GROUP# ────────────────────────────────────────────────────────────────────────────────print(f"\n" + "-"*60)print("B. PER STUDY GROUP")print("-"*60)print(f"\n  {'Study Group':<15} {'N':<10} {'Orig Score':<12} {'Adj Score':<12} {'Orig %SnC':<12} {'Adj %SnC':<12} {'Δ %SnC':<10}")print("  " + "-"*85)study_groups = sorted(adata.obs[STUDY_GROUP_COLUMN].unique())study_group_data = []for sg in study_groups:    sg_mask = adata.obs[STUDY_GROUP_COLUMN] == sg    sg_n = sg_mask.sum()    sg_orig_score = adata.obs.loc[sg_mask, SCORE_COL].mean()    sg_adj_score = adata.obs.loc[sg_mask, ADJUSTED_SCORE_COL].mean()    sg_orig_snc = adata.obs.loc[sg_mask, 'is_senescent'].sum()    sg_adj_snc = adata.obs.loc[sg_mask, 'is_senescent_adjusted'].sum()    sg_orig_pct = sg_orig_snc / sg_n * 100    sg_adj_pct = sg_adj_snc / sg_n * 100        study_group_data.append({        'Study_Group': sg, 'N': sg_n,        'Orig_Score': sg_orig_score, 'Adj_Score': sg_adj_score,        'Orig_SnC': sg_orig_snc, 'Adj_SnC': sg_adj_snc,        'Orig_Pct': sg_orig_pct, 'Adj_Pct': sg_adj_pct    })        print(f"  {sg:<15} {sg_n:<10,} {sg_orig_score:<12.4f} {sg_adj_score:<12.4f} "          f"{sg_orig_pct:<12.2f} {sg_adj_pct:<12.2f} {sg_adj_pct - sg_orig_pct:<+10.2f}")study_group_df = pd.DataFrame(study_group_data)# ────────────────────────────────────────────────────────────────────────────────# C. PER CELL TYPE# ────────────────────────────────────────────────────────────────────────────────print(f"\n" + "-"*60)print("C. PER CELL TYPE")print("-"*60)print(f"\n  {'Cell Type':<20} {'N':<10} {'Orig Score':<12} {'Adj Score':<12} {'Orig %SnC':<12} {'Adj %SnC':<12} {'Δ %SnC':<10}")print("  " + "-"*90)cell_types = sorted(adata.obs[CELL_TYPE_COLUMN].unique())cell_type_data = []for ct in cell_types:    ct_mask = adata.obs[CELL_TYPE_COLUMN] == ct    ct_n = ct_mask.sum()    ct_orig_score = adata.obs.loc[ct_mask, SCORE_COL].mean()    ct_adj_score = adata.obs.loc[ct_mask, ADJUSTED_SCORE_COL].mean()    ct_orig_snc = adata.obs.loc[ct_mask, 'is_senescent'].sum()    ct_adj_snc = adata.obs.loc[ct_mask, 'is_senescent_adjusted'].sum()    ct_orig_pct = ct_orig_snc / ct_n * 100    ct_adj_pct = ct_adj_snc / ct_n * 100        cell_type_data.append({        'Cell_Type': ct, 'N': ct_n,        'Orig_Score': ct_orig_score, 'Adj_Score': ct_adj_score,        'Orig_SnC': ct_orig_snc, 'Adj_SnC': ct_adj_snc,        'Orig_Pct': ct_orig_pct, 'Adj_Pct': ct_adj_pct    })        print(f"  {ct:<20} {ct_n:<10,} {ct_orig_score:<12.4f} {ct_adj_score:<12.4f} "          f"{ct_orig_pct:<12.2f} {ct_adj_pct:<12.2f} {ct_adj_pct - ct_orig_pct:<+10.2f}")cell_type_df = pd.DataFrame(cell_type_data)# ────────────────────────────────────────────────────────────────────────────────# D. PER STUDY GROUP × CELL TYPE# ────────────────────────────────────────────────────────────────────────────────print(f"\n" + "-"*60)print("D. PER STUDY GROUP × CELL TYPE")print("-"*60)sg_ct_data = []for sg in study_groups:    for ct in cell_types:        mask = (adata.obs[STUDY_GROUP_COLUMN] == sg) & (adata.obs[CELL_TYPE_COLUMN] == ct)        n = mask.sum()        if n == 0:            continue                orig_score = adata.obs.loc[mask, SCORE_COL].mean()        adj_score = adata.obs.loc[mask, ADJUSTED_SCORE_COL].mean()        orig_snc = adata.obs.loc[mask, 'is_senescent'].sum()        adj_snc = adata.obs.loc[mask, 'is_senescent_adjusted'].sum()                sg_ct_data.append({            'Study_Group': sg, 'Cell_Type': ct, 'N': n,            'Orig_Score': orig_score, 'Adj_Score': adj_score,            'Orig_SnC': orig_snc, 'Adj_SnC': adj_snc,            'Orig_Pct': orig_snc / n * 100,            'Adj_Pct': adj_snc / n * 100        })sg_ct_df = pd.DataFrame(sg_ct_data)# Pivot for displayprint("\n  Original %SnC:")pivot_orig = sg_ct_df.pivot(index='Cell_Type', columns='Study_Group', values='Orig_Pct')print(pivot_orig.round(1).to_string(index=True))print("\n  Adjusted %SnC:")pivot_adj = sg_ct_df.pivot(index='Cell_Type', columns='Study_Group', values='Adj_Pct')print(pivot_adj.round(1).to_string(index=True))print("\n  Δ %SnC (Adjusted - Original):")pivot_delta = pivot_adj - pivot_origprint(pivot_delta.round(1).to_string(index=True))# ────────────────────────────────────────────────────────────────────────────────# E. PER DONOR × STUDY GROUP × CELL TYPE# ────────────────────────────────────────────────────────────────────────────────print(f"\n" + "-"*60)print("E. PER DONOR × STUDY GROUP × CELL TYPE")print("-"*60)donor_data = []for donor in adata.obs[DONOR_COLUMN].unique():    donor_mask = adata.obs[DONOR_COLUMN] == donor    donor_sg = adata.obs.loc[donor_mask, STUDY_GROUP_COLUMN].iloc[0]        for ct in cell_types:        mask = donor_mask & (adata.obs[CELL_TYPE_COLUMN] == ct)        n = mask.sum()        if n == 0:            continue                orig_score = adata.obs.loc[mask, SCORE_COL].mean()        adj_score = adata.obs.loc[mask, ADJUSTED_SCORE_COL].mean()        orig_snc = adata.obs.loc[mask, 'is_senescent'].sum()        adj_snc = adata.obs.loc[mask, 'is_senescent_adjusted'].sum()                donor_data.append({            'Donor': donor,            'Study_Group': donor_sg,            'Cell_Type': ct,            'N': n,            'Orig_Score': orig_score,            'Adj_Score': adj_score,            'Orig_SnC': orig_snc,            'Adj_SnC': adj_snc,            'Orig_Pct': orig_snc / n * 100 if n > 0 else 0,            'Adj_Pct': adj_snc / n * 100 if n > 0 else 0        })donor_df = pd.DataFrame(donor_data)donor_df['Delta_Pct'] = donor_df['Adj_Pct'] - donor_df['Orig_Pct']# Summary statistics per study group × cell typeprint("\n  Donor-level summary (mean ± std of %SnC across donors):")print(f"\n  {'Study Group':<15} {'Cell Type':<15} {'N Donors':<10} {'Orig %SnC':<20} {'Adj %SnC':<20}")print("  " + "-"*80)donor_summary = donor_df.groupby(['Study_Group', 'Cell_Type']).agg({    'Donor': 'nunique',    'Orig_Pct': ['mean', 'std'],    'Adj_Pct': ['mean', 'std']}).reset_index()donor_summary.columns = ['Study_Group', 'Cell_Type', 'N_Donors',                           'Orig_Mean', 'Orig_Std', 'Adj_Mean', 'Adj_Std']for _, row in donor_summary.iterrows():    orig_str = f"{row['Orig_Mean']:.1f} ± {row['Orig_Std']:.1f}"    adj_str = f"{row['Adj_Mean']:.1f} ± {row['Adj_Std']:.1f}"    print(f"  {row['Study_Group']:<15} {row['Cell_Type']:<15} {int(row['N_Donors']):<10} "          f"{orig_str:<20} {adj_str:<20}")print(f"\n✓ Donor-level data shape: {donor_df.shape}")# ════════════════════════════════════════════════════════════════════════════════# FINAL SUMMARY# ════════════════════════════════════════════════════════════════════════════════print("\n" + "="*80)print("✓ UMI REGRESSION COMPLETE")print("="*80)print(f"\nNew columns added:")print(f"  adata.obs['{ADJUSTED_SCORE_COL}']")print(f"  adata.obs['is_senescent_adjusted']")print(f"  adata.obs['senescence_label_adjusted']")print(f"\nDataFrames created:")print(f"  regression_df: {regression_df.shape}")print(f"  study_group_df: {study_group_df.shape}")print(f"  cell_type_df: {cell_type_df.shape}")print(f"  sg_ct_df: {sg_ct_df.shape}")print(f"  donor_df: {donor_df.shape}")

In [ ]:
# ── source: module1p1.ipynb cell 17 ──# ════════════════════════════════════════════════════════════════════════════════# FIGURE: SnC BY STUDY GROUP PER CELL TYPE (Original vs Adjusted)# ════════════════════════════════════════════════════════════════════════════════print("\n" + "="*80)print("SnC BY STUDY GROUP PER CELL TYPE")print("="*80)# ─────────────────────────────────────────────────────────────────────────────────# Configuration# ─────────────────────────────────────────────────────────────────────────────────STUDY_TYPE = config['type']GROUP_ORDER_CONFIG = {    'psychad_aging': ['Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59',                       'Age_60_69', 'Age_70_79', 'Age_80_100'],    'psychad_aging_norm': ['Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59',                       'Age_60_69', 'Age_70_79', 'Age_80_100'],    'psychad_ad': ['Young_Healthy_Control', 'Old_Healthy_Control', 'Old_AD'],    'psychencode': ['Age_30_39', 'Age_40_49', 'Age_50_59',                     'Age_60_69', 'Age_70_79', 'Age_80_100'],    'psychencode_sub': ['Age_30_39', 'Age_40_49', 'Age_50_59',                         'Age_60_69', 'Age_70_79', 'Age_80_100'],    'mathys': ['NCI', 'MCI', 'AD'],}GROUP_ORDER = GROUP_ORDER_CONFIG.get(DATASET, [])STUDY_GROUP_COLORS = {    'Age_20_29': '#2E86AB', 'Age_30_39': '#4A90E2', 'Age_40_49': '#50C878',    'Age_50_59': '#FFB347', 'Age_60_69': '#FF8C00', 'Age_70_79': '#E24A4A',    'Age_80_100': '#8B0000',    'Control': '#4E79A7', 'MCI': '#F28E2B', 'AD': '#E15759', 'NCI': '#4E79A7',    'Young_Healthy_Control': '#4E79A7', 'Old_Healthy_Control': '#59A14F', 'Old_AD': '#E15759',}group_order = [g for g in GROUP_ORDER if g in donor_df['Study_Group'].unique()]ct_order = donor_df.groupby('Cell_Type')['Orig_Pct'].median().sort_values(ascending=False).index.tolist()n_celltypes = len(ct_order)n_cols = 4n_rows = int(np.ceil(n_celltypes / n_cols))if STUDY_TYPE == 'aging':    x_labels = [g.replace('Age_', '').replace('_', '–') for g in group_order]else:    x_labels = group_orderprint(f"Study type: {STUDY_TYPE}")print(f"Groups: {group_order}")print(f"Cell types: {ct_order}")# ─────────────────────────────────────────────────────────────────────────────────# PLOT 1: ORIGINAL LABELS# ─────────────────────────────────────────────────────────────────────────────────print("\n▸ Original labels:")fig, axes = plt.subplots(n_rows, n_cols, figsize=(6, 1.5 * n_rows), sharey=True)axes = axes.flatten()for idx, ct in enumerate(ct_order):    ax = axes[idx]    ct_df = donor_df[donor_df['Cell_Type'] == ct]        bp = ax.boxplot(        [ct_df[ct_df['Study_Group'] == g]['Orig_Pct'].values for g in group_order],        positions=range(len(group_order)),        widths=0.5, patch_artist=True, showfliers=False    )        for i, (box, group) in enumerate(zip(bp['boxes'], group_order)):        box.set_facecolor(STUDY_GROUP_COLORS.get(group, '#808080'))        box.set_alpha(0.7)        box.set_edgecolor('none')    for whisker in bp['whiskers']:        whisker.set_color('#666666')        whisker.set_linewidth(0.5)    for cap in bp['caps']:        cap.set_color('#666666')        cap.set_linewidth(0.5)    for median in bp['medians']:        median.set_color('#333333')        median.set_linewidth(0.8)        for i, group in enumerate(group_order):        group_data = ct_df[ct_df['Study_Group'] == group]['Orig_Pct'].values        if len(group_data) > 0:            jitter = np.random.uniform(-0.1, 0.1, size=len(group_data))            ax.scatter(np.repeat(i, len(group_data)) + jitter, group_data,                       c=STUDY_GROUP_COLORS.get(group, '#808080'), s=5, alpha=0.6,                        zorder=3, edgecolors='#333333', linewidths=0.2)        ax.set_title(ct, fontsize=7, fontweight='bold', pad=4)    ax.spines['top'].set_visible(False)    ax.spines['right'].set_visible(False)    ax.spines['left'].set_linewidth(0.5)    ax.spines['bottom'].set_linewidth(0.5)    ax.tick_params(labelsize=6, width=0.5)    ax.grid(False)    ax.set_xticks(range(len(group_order)))    ax.set_xticklabels(x_labels, rotation=90, ha='center', fontsize=6)    if idx % n_cols == 0:        ax.set_ylabel('SnC (%)', fontsize=7)for idx in range(n_celltypes, len(axes)):    axes[idx].set_visible(False)plt.suptitle(f'SnC by {STUDY_TYPE.capitalize()} Group per Cell Type (Original)',              fontsize=9, fontweight='bold', y=1.02)plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_group_celltype_original.svg', dpi=300, bbox_inches='tight', facecolor='white')plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_group_celltype_original.svg', dpi=150, bbox_inches='tight', facecolor='white')plt.show()print(f"✓ Saved: {DATASET}_snc_by_group_celltype_original.svg")# ─────────────────────────────────────────────────────────────────────────────────# PLOT 2: ADJUSTED LABELS# ─────────────────────────────────────────────────────────────────────────────────print("\n▸ Adjusted labels:")fig, axes = plt.subplots(n_rows, n_cols, figsize=(6, 1.5 * n_rows), sharey=True)axes = axes.flatten()for idx, ct in enumerate(ct_order):    ax = axes[idx]    ct_df = donor_df[donor_df['Cell_Type'] == ct]        bp = ax.boxplot(        [ct_df[ct_df['Study_Group'] == g]['Adj_Pct'].values for g in group_order],        positions=range(len(group_order)),        widths=0.5, patch_artist=True, showfliers=False    )        for i, (box, group) in enumerate(zip(bp['boxes'], group_order)):        box.set_facecolor(STUDY_GROUP_COLORS.get(group, '#808080'))        box.set_alpha(0.7)        box.set_edgecolor('none')    for whisker in bp['whiskers']:        whisker.set_color('#666666')        whisker.set_linewidth(0.5)    for cap in bp['caps']:        cap.set_color('#666666')        cap.set_linewidth(0.5)    for median in bp['medians']:        median.set_color('#333333')        median.set_linewidth(0.8)        for i, group in enumerate(group_order):        group_data = ct_df[ct_df['Study_Group'] == group]['Adj_Pct'].values        if len(group_data) > 0:            jitter = np.random.uniform(-0.1, 0.1, size=len(group_data))            ax.scatter(np.repeat(i, len(group_data)) + jitter, group_data,                       c=STUDY_GROUP_COLORS.get(group, '#808080'), s=5, alpha=0.6,                        zorder=3, edgecolors='#333333', linewidths=0.2)        ax.set_title(ct, fontsize=7, fontweight='bold', pad=4)    ax.spines['top'].set_visible(False)    ax.spines['right'].set_visible(False)    ax.spines['left'].set_linewidth(0.5)    ax.spines['bottom'].set_linewidth(0.5)    ax.tick_params(labelsize=6, width=0.5)    ax.grid(False)    ax.set_xticks(range(len(group_order)))    ax.set_xticklabels(x_labels, rotation=90, ha='center', fontsize=6)    if idx % n_cols == 0:        ax.set_ylabel('SnC (%)', fontsize=7)for idx in range(n_celltypes, len(axes)):    axes[idx].set_visible(False)plt.suptitle(f'SnC by {STUDY_TYPE.capitalize()} Group per Cell Type (Adjusted)',              fontsize=9, fontweight='bold', y=1.02)plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_group_celltype_adjusted.svg', dpi=300, bbox_inches='tight', facecolor='white')plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_group_celltype_adjusted.svg', dpi=150, bbox_inches='tight', facecolor='white')plt.show()print(f"✓ Saved: {DATASET}_snc_by_group_celltype_adjusted.svg")print("\n" + "="*80)print(f"✓ Cell types: {n_celltypes}")print(f"✓ Study groups: {len(group_order)}")print("="*80)

In [ ]:
# ── source: module1p1.ipynb cell 18 ──# ════════════════════════════════════════════════════════════════════════════════# FIGURE: SnC BY STUDY GROUP PER CELL TYPE - CELL LEVEL (faceted bar plot)# ════════════════════════════════════════════════════════════════════════════════print("\n" + "="*80)print("SnC BY STUDY GROUP PER CELL TYPE (CELL LEVEL)")print("="*80)# ─────────────────────────────────────────────────────────────────────────────────# Calculate cell-level SnC% for ORIGINAL labels# ─────────────────────────────────────────────────────────────────────────────────cell_level_orig = []for ct in adata.obs[CELL_TYPE_COLUMN].unique():    ct_mask = adata.obs[CELL_TYPE_COLUMN] == ct    ct_data = adata.obs[ct_mask]        for group in adata.obs[STUDY_GROUP_COLUMN].unique():        group_mask = ct_data[STUDY_GROUP_COLUMN] == group        group_data = ct_data[group_mask]        n_total = len(group_data)        if n_total > 0:            n_snc = group_data['is_senescent'].sum()            snc_pct = (n_snc / n_total * 100)            cell_level_orig.append({                'Cell_Type': ct,                'Study_Group': group,                'SnC_pct': snc_pct,                'n_cells': n_total            })df_cell_orig = pd.DataFrame(cell_level_orig)# ─────────────────────────────────────────────────────────────────────────────────# Calculate cell-level SnC% for ADJUSTED labels# ─────────────────────────────────────────────────────────────────────────────────cell_level_adj = []for ct in adata.obs[CELL_TYPE_COLUMN].unique():    ct_mask = adata.obs[CELL_TYPE_COLUMN] == ct    ct_data = adata.obs[ct_mask]        for group in adata.obs[STUDY_GROUP_COLUMN].unique():        group_mask = ct_data[STUDY_GROUP_COLUMN] == group        group_data = ct_data[group_mask]        n_total = len(group_data)        if n_total > 0:            n_snc = group_data['is_senescent_adjusted'].sum()            snc_pct = (n_snc / n_total * 100)            cell_level_adj.append({                'Cell_Type': ct,                'Study_Group': group,                'SnC_pct': snc_pct,                'n_cells': n_total            })df_cell_adj = pd.DataFrame(cell_level_adj)# ─────────────────────────────────────────────────────────────────────────────────# Setup# ─────────────────────────────────────────────────────────────────────────────────ct_order_cell = df_cell_orig.groupby('Cell_Type')['SnC_pct'].median().sort_values(ascending=False).index.tolist()group_order_cell = [g for g in GROUP_ORDER if g in df_cell_orig['Study_Group'].unique()]n_celltypes = len(ct_order_cell)n_cols = 4n_rows = int(np.ceil(n_celltypes / n_cols))if STUDY_TYPE == 'aging':    x_labels_cell = [g.replace('Age_', '').replace('_', '–') for g in group_order_cell]else:    x_labels_cell = group_order_cellprint(f"Cell types: {ct_order_cell}")print(f"Groups: {group_order_cell}")# ─────────────────────────────────────────────────────────────────────────────────# PLOT 1: ORIGINAL LABELS (CELL LEVEL)# ─────────────────────────────────────────────────────────────────────────────────print("\n▸ Original labels (cell level):")fig, axes = plt.subplots(n_rows, n_cols, figsize=(6, 1.5 * n_rows), sharey=True)axes = axes.flatten()for idx, ct in enumerate(ct_order_cell):    ax = axes[idx]    ct_df = df_cell_orig[df_cell_orig['Cell_Type'] == ct]        snc_values = []    colors = []    for group in group_order_cell:        group_data = ct_df[ct_df['Study_Group'] == group]        if len(group_data) > 0:            snc_values.append(group_data['SnC_pct'].values[0])        else:            snc_values.append(0)        colors.append(STUDY_GROUP_COLORS.get(group, '#808080'))        x_pos = np.arange(len(group_order_cell))    bars = ax.bar(x_pos, snc_values, color=colors, alpha=0.7, edgecolor='none')        for i, (bar, pct) in enumerate(zip(bars, snc_values)):        if pct > 0:            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,                    f'{pct:.1f}', ha='center', va='bottom', fontsize=4, rotation=90)        ax.set_title(ct, fontsize=7, fontweight='bold', pad=4)    ax.spines['top'].set_visible(False)    ax.spines['right'].set_visible(False)    ax.spines['left'].set_linewidth(0.5)    ax.spines['bottom'].set_linewidth(0.5)    ax.tick_params(labelsize=6, width=0.5)    ax.grid(False)    ax.set_xticks(x_pos)    ax.set_xticklabels(x_labels_cell, rotation=90, ha='center', fontsize=6)    if idx % n_cols == 0:        ax.set_ylabel('SnC (%)', fontsize=7)for idx in range(n_celltypes, len(axes)):    axes[idx].set_visible(False)plt.suptitle(f'SnC by {STUDY_TYPE.capitalize()} Group per Cell Type - Cell Level (Original)',              fontsize=9, fontweight='bold', y=1.02)plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_group_celltype_celllevel_original.svg', dpi=300, bbox_inches='tight', facecolor='white')plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_group_celltype_celllevel_original.svg', dpi=150, bbox_inches='tight', facecolor='white')plt.show()print(f"✓ Saved: {DATASET}_snc_by_group_celltype_celllevel_original.svg")# ─────────────────────────────────────────────────────────────────────────────────# PLOT 2: ADJUSTED LABELS (CELL LEVEL)# ─────────────────────────────────────────────────────────────────────────────────print("\n▸ Adjusted labels (cell level):")fig, axes = plt.subplots(n_rows, n_cols, figsize=(6, 1.5 * n_rows), sharey=True)axes = axes.flatten()for idx, ct in enumerate(ct_order_cell):    ax = axes[idx]    ct_df = df_cell_adj[df_cell_adj['Cell_Type'] == ct]        snc_values = []    colors = []    for group in group_order_cell:        group_data = ct_df[ct_df['Study_Group'] == group]        if len(group_data) > 0:            snc_values.append(group_data['SnC_pct'].values[0])        else:            snc_values.append(0)        colors.append(STUDY_GROUP_COLORS.get(group, '#808080'))        x_pos = np.arange(len(group_order_cell))    bars = ax.bar(x_pos, snc_values, color=colors, alpha=0.7, edgecolor='none')        for i, (bar, pct) in enumerate(zip(bars, snc_values)):        if pct > 0:            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,                    f'{pct:.1f}', ha='center', va='bottom', fontsize=4, rotation=90)        ax.set_title(ct, fontsize=7, fontweight='bold', pad=4)    ax.spines['top'].set_visible(False)    ax.spines['right'].set_visible(False)    ax.spines['left'].set_linewidth(0.5)    ax.spines['bottom'].set_linewidth(0.5)    ax.tick_params(labelsize=6, width=0.5)    ax.grid(False)    ax.set_xticks(x_pos)    ax.set_xticklabels(x_labels_cell, rotation=90, ha='center', fontsize=6)    if idx % n_cols == 0:        ax.set_ylabel('SnC (%)', fontsize=7)for idx in range(n_celltypes, len(axes)):    axes[idx].set_visible(False)plt.suptitle(f'SnC by {STUDY_TYPE.capitalize()} Group per Cell Type - Cell Level (Adjusted)',              fontsize=9, fontweight='bold', y=1.02)plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_group_celltype_celllevel_adjusted.svg', dpi=300, bbox_inches='tight', facecolor='white')plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_group_celltype_celllevel_adjusted.svg', dpi=150, bbox_inches='tight', facecolor='white')plt.show()print(f"✓ Saved: {DATASET}_snc_by_group_celltype_celllevel_adjusted.svg")print("\n" + "="*80)print(f"✓ Cell types: {n_celltypes}")print(f"✓ Groups: {len(group_order_cell)}")print(f"✓ Level: Cell (raw proportions, not donor-aggregated)")print("="*80)

In [ ]:
# ── source: module1p1.ipynb cell 19 ──# ════════════════════════════════════════════════════════════════════════════════# FIGURE: UMI vs SENESCENCE SCORE CORRELATION (Original & Adjusted Superimposed)# ════════════════════════════════════════════════════════════════════════════════print("\n" + "="*80)print("UMI vs SENESCENCE SCORE CORRELATION")print("="*80)# ─────────────────────────────────────────────────────────────────────────────────# Setup# ─────────────────────────────────────────────────────────────────────────────────cell_types = sorted(adata.obs[CELL_TYPE_COLUMN].unique())n_celltypes = len(cell_types)n_cols = 4n_rows = int(np.ceil(n_celltypes / n_cols))MAX_POINTS = 3000  # Per group, so 6000 total per panelprint(f"Cell types: {n_celltypes}")print(f"Max points per group: {MAX_POINTS}")# ─────────────────────────────────────────────────────────────────────────────────# SUPERIMPOSED PLOT: Original (blue) vs Adjusted (green)# ─────────────────────────────────────────────────────────────────────────────────fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 3 * n_rows))axes = axes.flatten()correlation_data = []for idx, ct in enumerate(cell_types):    ax = axes[idx]    ct_mask = adata.obs[CELL_TYPE_COLUMN] == ct    ct_data = adata.obs[ct_mask]        log_umi = np.log10(ct_data[UMI_COL].values)    scores_orig = ct_data[SCORE_COL].values    scores_adj = ct_data[ADJUSTED_SCORE_COL].values        # Remove invalid values    valid = np.isfinite(log_umi) & np.isfinite(scores_orig) & np.isfinite(scores_adj)    log_umi_valid = log_umi[valid]    scores_orig_valid = scores_orig[valid]    scores_adj_valid = scores_adj[valid]        # Correlations    r_orig, p_orig = stats.pearsonr(log_umi_valid, scores_orig_valid)    r_adj, p_adj = stats.pearsonr(log_umi_valid, scores_adj_valid)        correlation_data.append({        'Cell_Type': ct,         'r_orig': r_orig, 'p_orig': p_orig,        'r_adj': r_adj, 'p_adj': p_adj,        'n': len(log_umi_valid)    })        # Subsample for plotting    if len(log_umi_valid) > MAX_POINTS:        idx_sample = np.random.choice(len(log_umi_valid), MAX_POINTS, replace=False)        log_umi_plot = log_umi_valid[idx_sample]        scores_orig_plot = scores_orig_valid[idx_sample]        scores_adj_plot = scores_adj_valid[idx_sample]    else:        log_umi_plot = log_umi_valid        scores_orig_plot = scores_orig_valid        scores_adj_plot = scores_adj_valid        # Scatter - Original (blue, background)    ax.scatter(log_umi_plot, scores_orig_plot, alpha=0.1, s=2, c='#1f77b4',                rasterized=True, label='Original')        # Scatter - Adjusted (green, foreground)    ax.scatter(log_umi_plot, scores_adj_plot, alpha=0.1, s=2, c='#2ca02c',                rasterized=True, label='Adjusted')        # Regression lines    z_orig = np.polyfit(log_umi_valid, scores_orig_valid, 1)    z_adj = np.polyfit(log_umi_valid, scores_adj_valid, 1)    x_line = np.linspace(log_umi_valid.min(), log_umi_valid.max(), 100)        ax.plot(x_line, np.poly1d(z_orig)(x_line), '#1f77b4', linewidth=1.5, alpha=0.9)    ax.plot(x_line, np.poly1d(z_adj)(x_line), '#2ca02c', linewidth=1.5, alpha=0.9)        # Annotation (no border)    ax.text(0.05, 0.95, f'Orig: r={r_orig:.2f}\nAdj:  r={r_adj:.2f}',             transform=ax.transAxes, fontsize=7, va='top', ha='left',            fontfamily='monospace')        ax.set_title(ct, fontsize=9, fontweight='bold')    ax.set_xlabel('log₁₀(UMI)', fontsize=8)    ax.set_ylabel('Score', fontsize=8)    ax.spines['top'].set_visible(False)    ax.spines['right'].set_visible(False)    ax.tick_params(labelsize=7)    ax.grid(False)for idx in range(n_celltypes, len(axes)):    axes[idx].set_visible(False)# Legendhandles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#1f77b4', markersize=6, label='Original'),           plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ca02c', markersize=6, label='Adjusted')]fig.legend(handles=handles, loc='lower right', fontsize=8, frameon=False,            bbox_to_anchor=(0.98, 0.02))plt.suptitle('UMI vs Senescence Score: Original (blue) vs Adjusted (green)',              fontsize=11, fontweight='bold', y=1.01)plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_umi_vs_score_comparison.svg', dpi=300, bbox_inches='tight', facecolor='white')plt.savefig(FIGURES_DIR / f'{DATASET}_umi_vs_score_comparison.svg', dpi=150, bbox_inches='tight', facecolor='white')plt.show()print(f"✓ Saved: {DATASET}_umi_vs_score_comparison.svg")# ─────────────────────────────────────────────────────────────────────────────────# SUMMARY# ─────────────────────────────────────────────────────────────────────────────────corr_df = pd.DataFrame(correlation_data)corr_df['r_change'] = corr_df['r_adj'] - corr_df['r_orig']corr_df['r_reduction_pct'] = (1 - abs(corr_df['r_adj']) / abs(corr_df['r_orig'])) * 100print("\n" + "-"*60)print("CORRELATION SUMMARY")print("-"*60)print(f"\n  {'Cell Type':<20} {'r (Orig)':<12} {'r (Adj)':<12} {'Δr':<10} {'% Reduction':<12}")print("  " + "-"*66)for _, row in corr_df.iterrows():    print(f"  {row['Cell_Type']:<20} {row['r_orig']:<12.3f} {row['r_adj']:<12.3f} "          f"{row['r_change']:<+10.3f} {row['r_reduction_pct']:<12.1f}")print("  " + "-"*66)print(f"  {'MEAN':<20} {corr_df['r_orig'].mean():<12.3f} {corr_df['r_adj'].mean():<12.3f} "      f"{corr_df['r_change'].mean():<+10.3f} {corr_df['r_reduction_pct'].mean():<12.1f}")corr_df.to_csv(FIGURES_DIR / f'{DATASET}_umi_score_correlation.csv', index=False)print(f"\n✓ Saved: {DATASET}_umi_score_correlation.csv")print("\n" + "="*80)if corr_df['r_adj'].abs().mean() < 0.1:    print("✓ UMI confounding successfully removed (mean |r| < 0.1)")else:    print(f"⚠ Residual UMI correlation remains (mean |r| = {corr_df['r_adj'].abs().mean():.3f})")print("="*80)

## Validate with Canonical Markers**Why.** p16/p21 sanity check at the point of calling.<sub>source: `module1p1.ipynb` cell 21</sub>

In [ ]:
why("Validate with Canonical Markers", "p16/p21 sanity check at the point of calling")

In [ ]:
# ── source: module1p1.ipynb cell 21 ──print("\n" + "="*80)print("CANONICAL MARKER VALIDATION")print("="*80)# Canonical senescence markersSENESCENCE_MARKERS = ['CDKN1A', 'CDKN2A', 'TP53', 'CDKN2B', 'IL6', 'IL8', 'TREM2']# Check which markers are presentmarkers_present = [m for m in SENESCENCE_MARKERS if m in adata.var_names]markers_missing = [m for m in SENESCENCE_MARKERS if m not in adata.var_names]print(f"\nCanonical markers:")print(f"  Present: {markers_present}")print(f"  Missing: {markers_missing}")if len(markers_present) > 0:    # Compare expression in SnC vs Non-SnC    print(f"\nMarker expression (SnC vs Non-SnC):")        for marker in markers_present:        snc_expr = adata[adata.obs['is_senescent'], marker].X.mean()        nonsnc_expr = adata[~adata.obs['is_senescent'], marker].X.mean()        fold_change = snc_expr / nonsnc_expr if nonsnc_expr > 0 else 0                print(f"  {marker}:")        print(f"    SnC: {snc_expr:.3f}, Non-SnC: {nonsnc_expr:.3f}, FC: {fold_change:.2f}x")else:    print("\n⚠ No canonical markers found in dataset")print("\n✓ Validation complete")

## Visualizations**Why.** Score and SnC distribution across groups and cell types.<sub>source: `module1p1.ipynb` cells 23, 24, 25</sub>

In [ ]:
why("Visualizations", "Score and SnC distribution across groups and cell types")

In [ ]:
# ── source: module1p1.ipynb cell 23 ──# UMAP colored by senescence scorefig, axes = plt.subplots(1, 2, figsize=(14, 6))sc.pl.umap(adata, color='senescence_score', ax=axes[0], show=False,            title='Senescence Score', frameon=False, size=2, cmap='viridis')axes[0].set_xlabel('UMAP 1')axes[0].set_ylabel('UMAP 2')sc.pl.umap(adata, color='senescence_label', ax=axes[1], show=False,           title='SnC Classification', frameon=False, size=2, palette={'SnC': '#E63946', 'Non-SnC': 'lightgray'})axes[1].set_xlabel('UMAP 1')axes[1].set_ylabel('UMAP 2')plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_senescence_umap.svg', dpi=300, bbox_inches='tight')plt.show()print("✓ UMAP plots saved")

In [ ]:
# ── source: module1p1.ipynb cell 24 ──# SnC proportion by age group and cell typefig, ax = plt.subplots(figsize=(10, 6))# Calculate proportionsprop_data = []for age_group in sorted(adata.obs[STUDY_GROUP_COLUMN].unique()):    for cell_type in sorted(adata.obs[CELL_TYPE_COLUMN].unique()):        mask = (adata.obs[STUDY_GROUP_COLUMN] == age_group) & (adata.obs[CELL_TYPE_COLUMN] == cell_type)        if mask.sum() > 0:            snc_prop = adata.obs.loc[mask, 'is_senescent'].sum() / mask.sum() * 100            prop_data.append({'Age_Group': age_group, 'Cell_Type': cell_type, 'SnC_Proportion': snc_prop})prop_df = pd.DataFrame(prop_data)# Plot heatmappivot_df = prop_df.pivot(index='Cell_Type', columns='Age_Group', values='SnC_Proportion')sns.heatmap(pivot_df, annot=True, fmt='.1f', cmap='Reds',             cbar_kws={'label': 'SnC %'},             linewidths=0.5, linecolor='white',  # Thin white lines between cells            ax=ax)ax.set_title('SnC Proportion by Age Group and Cell Type', fontsize=11, pad=10)ax.set_xlabel('Age Group')ax.set_ylabel('Cell Type')# Remove gridax.grid(False)plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_age_celltype.svg', dpi=300, bbox_inches='tight')plt.show()print("✓ Heatmap saved")

In [ ]:
# ── source: module1p1.ipynb cell 25 ──# ═══════════════════════════════════════════════════════════════════════════════# UMAP: SnC HIGHLIGHTED BY STUDY GROUP# ═══════════════════════════════════════════════════════════════════════════════%matplotlib inlineprint("\n" + "="*80)print("UMAP: SENESCENT CELLS BY STUDY GROUP")print("="*80)available_groups = [g for g in GROUP_ORDER if g in adata.obs['Study_Group'].unique()]n_groups = len(available_groups)n_cols = 4n_rows = int(np.ceil(n_groups / n_cols))umap_key = 'X_umap'if umap_key not in adata.obsm:    print("  ✗ UMAP not found in adata.obsm")else:    umap_coords = adata.obsm[umap_key]        fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 3.5 * n_rows))    axes = axes.flatten()        for idx, group in enumerate(available_groups):        ax = axes[idx]                group_mask = (adata.obs['Study_Group'] == group).values        snc_mask = (adata.obs['is_senescent'] == True).values                # Non-SnC (gray)        non_snc = group_mask & ~snc_mask        ax.scatter(umap_coords[non_snc, 0], umap_coords[non_snc, 1],                   s=0.3, c='#E0E0E0', alpha=0.3, rasterized=True)                # SnC (red)        snc_in_group = group_mask & snc_mask        ax.scatter(umap_coords[snc_in_group, 0], umap_coords[snc_in_group, 1],                   s=0.5, c='#E15759', alpha=0.5, rasterized=True)                # Stats        n_total = group_mask.sum()        n_snc = snc_in_group.sum()        pct = n_snc / n_total * 100 if n_total > 0 else 0                group_label = group.replace('Age_', '').replace('_', '–')        ax.set_title(f'{group_label}\n({n_snc:,} SnC / {n_total:,}, {pct:.1f}%)',                     fontsize=7, fontweight='bold', pad=3)                ax.set_xticks([])        ax.set_yticks([])        ax.spines['top'].set_visible(False)        ax.spines['right'].set_visible(False)        ax.spines['left'].set_visible(False)        ax.spines['bottom'].set_visible(False)                # UMAP arrows per panel        xlim = ax.get_xlim()        ylim = ax.get_ylim()        x_start = xlim[0] + (xlim[1] - xlim[0]) * 0.02        y_start = ylim[0] + (ylim[1] - ylim[0]) * 0.02        arrow_x = (xlim[1] - xlim[0]) * 0.12        arrow_y = (ylim[1] - ylim[0]) * 0.12                ax.annotate('', xy=(x_start + arrow_x, y_start), xytext=(x_start, y_start),                     arrowprops=dict(arrowstyle='->', lw=0.6, color='black'))        ax.annotate('', xy=(x_start, y_start + arrow_y), xytext=(x_start, y_start),                     arrowprops=dict(arrowstyle='->', lw=0.6, color='black'))        ax.text(x_start + arrow_x / 2, y_start - (ylim[1] - ylim[0]) * 0.03,                'UMAP1', fontsize=4, ha='center', va='top', fontweight='bold')        ax.text(x_start - (xlim[1] - xlim[0]) * 0.03, y_start + arrow_y / 2,                'UMAP2', fontsize=4, ha='right', va='center', rotation=90, fontweight='bold')        for idx in range(n_groups, len(axes)):        axes[idx].set_visible(False)        # Legend    from matplotlib.lines import Line2D    legend_elements = [        Line2D([0], [0], marker='o', color='w', markerfacecolor='#E0E0E0',               markersize=5, label='Non-SnC'),        Line2D([0], [0], marker='o', color='w', markerfacecolor='#E15759',               markersize=5, label='SnC'),    ]    fig.legend(handles=legend_elements, loc='lower right',               bbox_to_anchor=(0.98, 0.02), fontsize=7, frameon=False)        plt.suptitle(f'{DATASET}: Senescent Cells by Study Group',                 fontsize=10, fontweight='bold', y=1.01)    plt.tight_layout()    plt.savefig(FIGURES_DIR / f'{DATASET}_umap_snc_by_studygroup.pdf', dpi=300, bbox_inches='tight')    plt.savefig(FIGURES_DIR / f'{DATASET}_umap_snc_by_studygroup.svg', dpi=150, bbox_inches='tight')    plt.show()        print(f"\n✓ Saved: {DATASET}_umap_snc_by_studygroup.pdf")

## Save Results**Why.** Writes `{dataset}_pearson_senescence_scored.h5ad`.<sub>source: `module1p1.ipynb` cells 27, 28</sub>

In [ ]:
why("Save Results", "Writes `{dataset}_pearson_senescence_scored")

In [ ]:
# ── source: module1p1.ipynb cell 27 ──# ════════════════════════════════════════════════════════════════════════════════# SAVE CURRENT .X AS LAYER & SET EXPRESSION SOURCE# ════════════════════════════════════════════════════════════════════════════════# Preserve current .X (Pearson residuals) as a named layerif 'regressed_counts' not in adata.layers:    adata.layers['regressed_counts'] = adata.X.copy()    print("  ✓ Saved current .X → adata.layers['regressed_counts']")else:    print("  • regressed_counts layer already exists")# Activate lognorm as .X for downstream analysisadata.X = adata.layers['lognorm'].copy()print("  ✓ Activated adata.layers['lognorm'] → adata.X")print(f"  Layers: {list(adata.layers.keys())}")print(f"  .X range: [{adata.X.min():.2f}, {adata.X.max():.2f}]")

In [ ]:
# ── source: module1p1.ipynb cell 28 ──print("\n" + "="*80)print("SAVING")print("="*80)# Save as subset-specific file to preserve the full scored datasetOUTPUT_FILE_SUBSET = OUTPUT_FILE.parent / f'{DATASET}_pearson_senescence_scored.h5ad'OUTPUT_FILE_SUBSET.parent.mkdir(parents=True, exist_ok=True)print(f"\nSaving to: {OUTPUT_FILE_SUBSET}")print(f"  (Full dataset preserved at: {OUTPUT_FILE.name})")adata.write_h5ad(OUTPUT_FILE_SUBSET)file_size = OUTPUT_FILE_SUBSET.stat().st_size / 1e9print(f"\n✓ Saved ({file_size:.2f} GB)")print(f"  Cells: {adata.n_obs:,}")print(f"  Genes: {adata.n_vars:,}")print(f"  Cell types: {adata.obs[CELL_TYPE_COLUMN].nunique()}")print(f"  SnC cells: {adata.obs['is_senescent'].sum():,} ({adata.obs['is_senescent'].sum()/adata.n_obs*100:.1f}%)")

## Summary**Why.** Run record.<sub>source: `module1p1.ipynb` cell 30</sub>

In [ ]:
why("Summary", "Run record")

In [ ]:
# ── source: module1p1.ipynb cell 30 ──print("\n" + "="*80)print("✓ SENESCENCE SCORING COMPLETE")print("="*80)print(f"\nDataset: {DATASET}")print(f"\nScoring method:")print(f"  Method: SenePy (hippocampus, cell-type and sex-specific)")print(f"  Threshold: Mean + {SD_THRESHOLD} SD from {REFERENCE_GROUP}")print(f"\nFinal dimensions:")print(f"  Cells: {adata.n_obs:,}")print(f"  Genes: {adata.n_vars:,}")print(f"  Age groups: {adata.obs[STUDY_GROUP_COLUMN].nunique()}")print(f"  Cell types: {adata.obs[CELL_TYPE_COLUMN].nunique()}")print(f"\nSenescence classification:")print(f"  SnC (senescent): {adata.obs['is_senescent'].sum():,} ({adata.obs['is_senescent'].sum()/adata.n_obs*100:.1f}%)")print(f"  Non-SnC: {(~adata.obs['is_senescent']).sum():,} ({(~adata.obs['is_senescent']).sum()/adata.n_obs*100:.1f}%)")print(f"\nNew columns added:")print(f"  adata.obs['Study_Group']: harmonized age/diagnosis groups")print(f"  adata.obs['senescence_score']: continuous score")print(f"  adata.obs['is_senescent']: binary classification (True/False)")print(f"  adata.obs['senescence_label']: binary classification (SnC/Non-SnC)")print(f"\nValidation:")print(f"  Canonical markers enriched in SnC cells:")print(f"    CDKN1A: 3.8x, TP53: 3.6x, IL6: 3.3x")print(f"\nOutput files:")print(f"  Data: {OUTPUT_FILE}")print(f"  Figures: {FIGURES_DIR}/")print("\n" + "="*80)print("✓ Ready for Module 02: Statistical Analysis (Aging)")print("="*80)

## GATE**Why.** `is_senescent` is the single most consumed variable in the pipeline. This gate records the realized SnC fraction and the score-vs-depth coupling rather than asserting a target value — the realized fraction is a property of the data, and tuning `SD_THRESHOLD` to hit a number would convert an anchored SD threshold into a quantile threshold.

In [ ]:
# ⟨NEW⟩ exit gateimport numpy as npfrom scipy.stats import spearmanrwhy("03 GATE", "Are the senescence calls usable downstream?")gate("is_senescent present", 'is_senescent' in adata.obs)gate("senescence_score present", 'senescence_score' in adata.obs)gate("threshold used reference group only",     (adata.obs[C.STUDY_GROUP_COLUMN] == cfg.reference_group).sum() > 0,     f"reference={cfg.reference_group}")_pct = 100 * adata.obs['is_senescent'].mean()print(f"\n  realized SnC   : {_pct:.2f}%   (RECORD this - do not tune SD_THRESHOLD to it)")_r = spearmanr(adata.obs['senescence_score'], adata.obs['total_counts']).statisticprint(f"  score ~ depth  : rho = {_r:+.3f}")print("  Reported, not corrected. Senescent cells are larger and transcriptionally")print("  hyperactive, so elevated depth is expected biology. Module 04 carries the")print("  depth-matched downsampling sensitivity analysis that defends this choice.")gate("adjusted score not consumed downstream",     'senescence_score_adjusted' in adata.obs,     "present as a diagnostic only; no downstream module reads it")for _ct, _g in adata.obs.groupby(cfg.cell_type_column):    _n = int(_g['is_senescent'].sum())    if _n < C.MIN_SENESCENT_EVENTS:        print(f"  [WARN] {_ct}: {_n} senescent events "              f"(< MIN_SENESCENT_EVENTS={C.MIN_SENESCENT_EVENTS}) -> not estimable downstream")